# H002 — رأس التصنيف الثنائي في `NIG-TimeNet v2`

هذا الدفتر فرضية واحدة محدَّدة — يعتمد على دفتر المحور الأساسي `signal_evaluation_axis (3).ipynb` (IC، فحص عُشر، خطّ أساس عشوائي، `evaluate_windows`، وسجلّ التجارب `register_hypothesis`/`list_registry`) بدل تكراره. شغّل الخلية التالية أولاً (أو تأكد من تشغيل الدفتر الأساسي في نفس الجلسة) قبل أي خلية أخرى هنا.

أُفرِد هذا الملف من `signal_evaluation_axis (3).ipynb` الأصلي — نفس المحتوى حرفياً، بلا أي تعديل على الأرقام أو المنطق.

In [ ]:
%run "signal_evaluation_axis (3).ipynb"

## ١٥) H002 — رأس التصنيف الثنائي في `NIG-TimeNet v2` (اتجاه يوم واحد)

تفعيل رؤوس التصنيف في `model_v2` (`MODEL_CONFIG['head_types']`) أثار سؤالاً
مباشراً: هل احتمال الاتجاه الذي يُخرجه رأس `binary_classification` يحمل
إشارة حقيقية، أم أنه — كما بدا من تدريب أوّلي قصير (5 حقب) على بيانات
حقيقية في `main.ipynb` — قريب من الصدفة (دقّة 46-67%، AUC مشابه)؟ التدريب
القصير وحده لا يحسم: هل السقف القريب من 50% سببه ضعف الإشارة في الميزات،
أم قصر التدريب نفسه (رؤوس الانحدار على نفس الجذع تحسّنت بوضوح خلال نفس
الحقب الخمس، بينما التصنيف بقي عالقاً — أنظف طريقة للحسم هي محور هذا
الدفتر بالضبط: IC + عُشر + خطّ أساس عشوائي عبر نوافذ متحرّكة، بنفس صرامة H001.

### الإعداد (يختلف عن `momentum_predict_fn` — تدريب لا صيغة ثابتة)

`H001` استخدمت فرضية بلا تدريب إطلاقاً (آخر تغيّر سعري) — لا ضجيج من تقدير
نموذج يتداخل مع قياس الإشارة نفسها. رأس تصنيف NIG-TimeNet v2 يحتاج تدريباً
بالضرورة، فـ`predict_fn` هنا **يُدرِّب نموذجاً مستقلاً من الصفر لكل نافذة**
(بيانات تلك النافذة فقط، 8 حقب، بلا استئناف بين النوافذ) ثم يُخرج احتمال
الاتجاه (`sigmoid`) على `test` — يقيس هذا **قدرة الإعداد الحالي** (بيانات
محدودة + تدريب قصير لكل نافذة) على استخلاص إشارة، لا سقف المعمارية نظرياً
بتدريب كافٍ (فرق جوهري يجب عدم تجاهله عند قراءة النتيجة أدناه).

بيانات حقيقية فعلية — 5 عملات من `history_1d` (SOLUSDT منذ 2020، ILVUSDT،
PYTHUSDT، RPLUSDT، DRIFTUSDT) — لا تركيبية. `rolling_splits(dataset,
test_span='30D', val_span='15D', initial_train_span='365D', step='30D',
max_windows=12)`، `min_split_samples` مخفَّض إلى 10 مؤقّتاً (نوافذ مبكّرة
بأصل واحد نشط فقط لا تبلغ 64 عيّنة بامتداد 15-30 يوماً — 10 هو الحدّ الذي
تفرضه `compute_ic` نفسها، لا حدّ تعسّفي أدنى).

الاحتمال (لا التسمية الثنائية بعد العتبة) يُقارَن عبر IC سبيرمان بـ`y_close_reg`/
`y_high_reg`/`y_low_reg` (العائد الفعلي المستمر لكل هدف) — وليس بالتسمية
`{0,1}` نفسها: مقياس أدقّ لجودة الترتيب الذي ينتجه الرأس، بنفس منطق تقييم
H001 لـ`momentum_predict_fn`.

### النتيجة — لا إشارة موثوقة عبر 12 نافذة (الثلاثة أهداف)

| الهدف | IC متوسط | انحراف IC | اتّساق الإشارة | نوافذ معنوية بنفس الاتجاه |
|---|---|---|---|---|
| `close` | -0.038 | 0.235 | **لا** | 8% (1/12) |
| `high` | -0.061 | 0.172 | **لا** | 0% (0/12) |
| `low` | -0.146 | 0.255 | **لا** | 25% (3/12) |

الإشارة الأهم: **علامة IC تنقلب باستمرار بين النوافذ** (مثلاً `close`:
+0.41 في النافذة 5، ‑0.39 في النافذة 3؛ `low`: +0.38 في النافذة 3، ‑0.52 في
النافذة 8) — هذا بالضبط ما حذّرت منه فلسفة هذا المحور: نافذة واحدة قوية لا
تعني شيئاً بلا اتّساق الاتجاه عبر نوافذ متعدّدة (راجع توثيق `evaluate_windows`).
لا هدف من الثلاثة حقّق `consistent_sign=True`.

### القرار: مرفوضة — بهذا الإعداد تحديداً

**H002 (بصيغتها المُختبَرة هنا: نموذج مُدرَّب من الصفر لكل نافذة، 5 أصول،
8 حقب) لا تُظهر إشارة اتجاه موثوقة.** هذا يتّفق مع IC الضعيف أصلاً
لأقوى فرضية مقبولة حالياً (H001، انعكاس قصير المدى، IC≈-0.08) — سقف قريب
من العملة المعدنية العادلة متوقَّع لاتجاه يوم واحد من ميزات OHLCV+مؤشرات.

**سؤال مفتوح لم يُحسَم (بالضبط كشذوذ 14/18 في H001 — موثَّق لا مُغلَق زوراً):**
هل انعدام الاتساق سببه ضعف الإشارة فعلاً، أم أن 8 حقب/365-695 عيّنة لكل
نافذة غير كافية إطلاقاً لتقارب نموذج NIG-TimeNet v2 (~مئات آلاف المعاملات)
بشكل يعتدّ به إحصائياً — تباين تقدير النموذج نفسه قد يهيمن على تباين IC
المقيس؟ **لا يحسم هذا الاختبار بين الاحتمالين.** يحتاج حسمه: نفس المحور
بنموذج مُدرَّب تدريباً كاملاً (60 حقبة، كما في `main.ipynb`) على مجموعة
أصول أكبر (422 عملة، مثل بقية تجارب المشروع) — تكلفة حسابية أكبر بكثير،
تُترَك لتشغيل لاحق صريح.


In [ ]:
# @title
# النتائج الفعلية أدناه (تشغيل حقيقي: 5 أصول من history_1d، rolling_splits
# بـ12 نافذة، نموذج NIG-TimeNet v2 مُدرَّب من الصفر لكل نافذة — 8 حقب) —
# مُضمَّنة حرفياً هنا (لا الدفتر بلا بيانات Drive حقيقية وقت الكتابة، بنفس
# نمط تسجيل H001 أعلاه)، لتبقى الخلية قابلة لإعادة القراءة/الفحص لاحقاً
# بلا الحاجة لإعادة تشغيل التدريب الكامل (~5 دقائق لكل تشغيل).

_h002_per_window = {
    "close": [
        {"window": "نافذة 1", "status": "ok", "ic": -0.215573, "spread": -0.101748, "monotonic": False, "p_value": 0.251, "percentile": 12.8},
        {"window": "نافذة 2", "status": "ok", "ic": -0.086096, "spread": -0.013678, "monotonic": False, "p_value": 0.647, "percentile": 32.2},
        {"window": "نافذة 3", "status": "ok", "ic": -0.385095, "spread": -0.078799, "monotonic": False, "p_value": 0.039, "percentile": 1.8},
        {"window": "نافذة 4", "status": "ok", "ic": 0.094994, "spread": 0.011853, "monotonic": False, "p_value": 0.583, "percentile": 72.1},
        {"window": "نافذة 5", "status": "ok", "ic": 0.413126, "spread": 0.049807, "monotonic": False, "p_value": 0.023, "percentile": 98.8},
        {"window": "نافذة 6", "status": "ok", "ic": -0.112792, "spread": -0.029114, "monotonic": False, "p_value": 0.543, "percentile": 27.7},
        {"window": "نافذة 7", "status": "ok", "ic": -0.290323, "spread": 0.001024, "monotonic": False, "p_value": 0.118, "percentile": 5.1},
        {"window": "نافذة 8", "status": "ok", "ic": -0.183982, "spread": -0.028616, "monotonic": False, "p_value": 0.330, "percentile": 16.0},
        {"window": "نافذة 9", "status": "ok", "ic": -0.190656, "spread": 0.023237, "monotonic": False, "p_value": 0.322, "percentile": 15.9},
        {"window": "نافذة 10", "status": "ok", "ic": 0.311235, "spread": -0.150736, "monotonic": False, "p_value": 0.095, "percentile": 95.8},
        {"window": "نافذة 11", "status": "ok", "ic": 0.189766, "spread": 0.060514, "monotonic": False, "p_value": 0.307, "percentile": 85.3},
        {"window": "نافذة 12", "status": "ok", "ic": -0.000222, "spread": 0.066174, "monotonic": False, "p_value": 1.000, "percentile": 53.0},
    ],
    "high": [
        {"window": "نافذة 1", "status": "ok", "ic": -0.021580, "spread": 0.017920, "monotonic": False, "p_value": 0.905, "percentile": 43.8},
        {"window": "نافذة 2", "status": "ok", "ic": 0.117686, "spread": 0.003572, "monotonic": False, "p_value": 0.539, "percentile": 74.9},
        {"window": "نافذة 3", "status": "ok", "ic": -0.169744, "spread": -0.010518, "monotonic": False, "p_value": 0.386, "percentile": 18.0},
        {"window": "نافذة 4", "status": "ok", "ic": 0.034038, "spread": 0.043189, "monotonic": False, "p_value": 0.861, "percentile": 55.3},
        {"window": "نافذة 5", "status": "ok", "ic": -0.352614, "spread": -0.067815, "monotonic": False, "p_value": 0.053, "percentile": 2.6},
        {"window": "نافذة 6", "status": "ok", "ic": 0.000667, "spread": 0.013121, "monotonic": False, "p_value": 0.997, "percentile": 52.0},
        {"window": "نافذة 7", "status": "ok", "ic": -0.229366, "spread": 0.006243, "monotonic": False, "p_value": 0.202, "percentile": 9.0},
        {"window": "نافذة 8", "status": "ok", "ic": -0.234705, "spread": -0.028492, "monotonic": False, "p_value": 0.215, "percentile": 11.8},
        {"window": "نافذة 9", "status": "ok", "ic": -0.257397, "spread": -0.041286, "monotonic": False, "p_value": 0.161, "percentile": 8.4},
        {"window": "نافذة 10", "status": "ok", "ic": 0.205784, "spread": 0.010503, "monotonic": False, "p_value": 0.279, "percentile": 86.9},
        {"window": "نافذة 11", "status": "ok", "ic": 0.071413, "spread": 0.039336, "monotonic": False, "p_value": 0.726, "percentile": 64.5},
        {"window": "نافذة 12", "status": "ok", "ic": 0.099889, "spread": 0.009949, "monotonic": False, "p_value": 0.585, "percentile": 72.0},
    ],
    "low": [
        {"window": "نافذة 1", "status": "ok", "ic": 0.204004, "spread": 0.052959, "monotonic": False, "p_value": 0.258, "percentile": 87.6},
        {"window": "نافذة 2", "status": "ok", "ic": -0.020245, "spread": -0.022943, "monotonic": False, "p_value": 0.917, "percentile": 45.1},
        {"window": "نافذة 3", "status": "ok", "ic": 0.377086, "spread": 0.058506, "monotonic": False, "p_value": 0.038, "percentile": 98.4},
        {"window": "نافذة 4", "status": "ok", "ic": 0.024249, "spread": 0.119567, "monotonic": False, "p_value": 0.899, "percentile": 54.3},
        {"window": "نافذة 5", "status": "ok", "ic": -0.427809, "spread": -0.106386, "monotonic": False, "p_value": 0.013, "percentile": 0.6},
        {"window": "نافذة 6", "status": "ok", "ic": -0.148387, "spread": 0.033032, "monotonic": False, "p_value": 0.449, "percentile": 22.3},
        {"window": "نافذة 7", "status": "ok", "ic": -0.277419, "spread": -0.018816, "monotonic": False, "p_value": 0.134, "percentile": 6.8},
        {"window": "نافذة 8", "status": "ok", "ic": -0.515907, "spread": -0.050724, "monotonic": False, "p_value": 0.006, "percentile": 0.3},
        {"window": "نافذة 9", "status": "ok", "ic": -0.111012, "spread": -0.020221, "monotonic": False, "p_value": 0.572, "percentile": 28.3},
        {"window": "نافذة 10", "status": "ok", "ic": -0.187987, "spread": -0.159548, "monotonic": False, "p_value": 0.319, "percentile": 14.4},
        {"window": "نافذة 11", "status": "ok", "ic": -0.197775, "spread": -0.064843, "monotonic": False, "p_value": 0.298, "percentile": 14.3},
        {"window": "نافذة 12", "status": "ok", "ic": -0.466518, "spread": -0.121235, "monotonic": False, "p_value": 0.014, "percentile": 0.9},
    ],
}

_h002_summary = {
    "close": {"mean_ic": -0.0380, "std_ic": 0.2353, "frac_significant": 0.0833, "consistent_sign": False, "n_ok": 12},
    "high": {"mean_ic": -0.0613, "std_ic": 0.1720, "frac_significant": 0.0000, "consistent_sign": False, "n_ok": 12},
    "low": {"mean_ic": -0.1456, "std_ic": 0.2554, "frac_significant": 0.2500, "consistent_sign": False, "n_ok": 12},
}

_h002_report = {
    "per_window": pd.DataFrame(_h002_per_window["close"]),
    **_h002_summary["close"],
}

register_hypothesis(
    hyp_id="H002_nig_timenet_classification_head",
    hypothesis=(
        "رأس التصنيف الثنائي (binary_classification، sigmoid) في NIG-TimeNet v2 "
        "(احتمال اتجاه يوم واحد صاعد/هابط لكل من high/low/close) يحمل إشارة "
        "حقيقية موثوقة عبر الزمن — يُختبَر عبر IC سبيرمان بين الاحتمال المُخرَج "
        "والعائد الفعلي المستمر لنفس الهدف، لا التسمية الثنائية بعد العتبة."
    ),
    source="hypothesis_driven",
    status="مرفوضة",
    report=_h002_report,
    notes=(
        "مرفوضة بهذا الإعداد تحديداً — لا كحكم نهائي على المعمارية. الإعداد: "
        "5 أصول حقيقية من history_1d (SOLUSDT منذ 2020، ILVUSDT، PYTHUSDT، "
        "RPLUSDT، DRIFTUSDT)، rolling_splits (test_span=30D, val_span=15D, "
        "initial_train_span=365D, step=30D)، 12 نافذة، نموذج مُدرَّب من الصفر "
        "لكل نافذة (8 حقب، بلا استئناف). أهم النتائج الثلاثة (close/high/low): "
        "mean_ic={-0.038,-0.061,-0.146}، consistent_sign=False والثلاثة معاً "
        "(العلامة تنقلب بين النوافذ — close: +0.41 نافذة 5 مقابل -0.39 نافذة 3؛ "
        "low: +0.38 نافذة 3 مقابل -0.52 نافذة 8)، نوافذ معنوية بنفس اتجاه "
        "المتوسط: {8%,0%,25%}. هذا اتساقه أضعف بكثير من H001 (22/30 نافذة "
        "بنفس الاتجاه). لا هدف حقّق consistent_sign=True — بالضبط ما تُصمَّم "
        "evaluate_windows لرصده كدليل عدم موثوقية. **سؤال مفتوح غير مُفسَّر "
        "(كشذوذ 14/18 في H001 — موثَّق لا مُغلَق زوراً)**: التدريب من الصفر لكل "
        "نافذة على 365-695 عيّنة فقط و8 حقب قد لا يكفي إطلاقاً لتقارب نموذج "
        "NIG-TimeNet v2 (رؤوس الانحدار على نفس الجذع تحسّنت بوضوح في تدريب "
        "main.ipynb الأولي بنفس عدد الحقب القليل، بينما التصنيف بقي عالقاً — "
        "يرجّح أن تباين تقدير النموذج نفسه، لا غياب الإشارة فقط، يساهم في "
        "تذبذب IC هنا). التوصية: إعادة الاختبار بنموذج مُدرَّب تدريباً كاملاً "
        "(60 حقبة كإعداد main.ipynb الفعلي) على مجموعة أصول أكبر (422 عملة، "
        "كبقية تجارب المشروع) قبل اعتبار السؤال محسوماً نهائياً — تكلفة "
        "حسابية أكبر بكثير من هذا الاختبار الأوّلي، تُترَك لتشغيل لاحق صريح."
    ),
)

list_registry()


## ١٦) تحقّق: هل تشابه العيّنات بين الأصول (نفس الفترة) يفسّر تذبذب IC في H002؟

سؤال مباشر على H002: بما أن `train_ds` يُخلَط (`tf.data.Dataset.shuffle`) قبل
التدريب، وبما أن عيّنات أصول مختلفة لنفس التاريخ متجاورة داخل `dataset`، هل
هذا الخلط يجعل التنبؤ "أسهل" بشكل مصطنع — يرى النموذج عيّنة عملة أخرى لنفس
الفترة فيستغلّ تشابهاً عاماً (الاتجاه السوقي المشترك) بدل تعلّم إشارة حقيقية
خاصة بكل أصل؟ هذا يستحقّ تحقّقاً منفصلاً عن التدريب القصير الذي وثّقناه أعلاه
— ليس نفس السؤال.

### أ) تسرّب بيانات حرفي؟ لا — تأكّد من الكود مباشرة

* `split_data`/`rolling_splits` (دفتر التحضير) تُقسِّمان بحدود زمنية **مطلقة
  مشتركة** بين كل الأصول (`mode='global_time'`) مع **فجوة عزل** (embargo) بين
  كل قسمين — لا خلط إطلاقاً قبل أو أثناء التقسيم، والحدّ الزمني يمنع أي عيّنة
  اختبار من الوقوع قبل عيّنة تدريب لنفس الأصل أو غيره.
* الخلط (`shuffle(4096)`) في `main.ipynb`/سكربت هذا الاختبار يُطبَّق **فقط**
  على `train_ds` بعد أن يخرج من التقسيم الزمني أصلاً — يُعيد ترتيب دفعات
  التدريب فقط، لا يُحرّك عيّنة واحدة عبر حدّ train/val/test.
* بحث مباشر في `model_v2` عن `BatchNormalization` أو أي طبقة تعتمد على
  إحصاءات الدُّفعة (batch statistics): **لا وجود لها** — كل الطبقات
  (`RMSNorm`, `RevIN`-style instance norm, `LayerNorm` ضمنياً عبر Transformer)
  تُطبَّع كل عيّنة بمعزل عن بقية عيّنات دفعتها. فلا آلية تقنية تسمح لعيّنة
  بـ"رؤية" عيّنة أخرى في نفس الدُّفعة أثناء التمرير الأمامي أو حساب الخسارة،
  مهما تشابهت الدفعة تاريخياً.

**الخلاصة: لا تسرّب بيانات حرفي بسبب الخلط.** لكن هذا لا يُغلق السؤال —
الجزء الأهمّ لا يزال مفتوحاً: هل الأصول الخمسة نفسها مستقلّة إحصائياً؟

### ب) الارتباط الفعلي بين الأصول — هنا الجزء المُقلِق فعلاً

قِسنا الارتباط (Spearman) بين `y_close_reg` (العائد الفعلي التالي) لكل زوج من
الأصول الخمسة في بيانات H002، بمحاذاة كل عيّنتين بتاريخهما (لا ترتيبهما في
المصفوفة)، وأيضاً معدّل توافق الاتجاه (`y_close_class` ∈ {+1,-1}) على نفس
التواريخ المشتركة:

| | DRIFT | ILV | PYTH | RPL | SOL |
|---|---|---|---|---|---|
| **DRIFT** | 1.00 | 0.56 | 0.55 | 0.54 | 0.60 |
| **ILV** | 0.56 | 1.00 | 0.70 | 0.71 | 0.68 |
| **PYTH** | 0.55 | 0.70 | 1.00 | 0.68 | 0.73 |
| **RPL** | 0.54 | 0.71 | 0.68 | 1.00 | 0.65 |
| **SOL** | 0.60 | 0.68 | 0.73 | 0.65 | 1.00 |

متوسط الارتباط الزوجي (خارج القطر) = **0.64** — قوي جداً لعملات مختلفة تماماً
في القطاع والسيولة. معدّل توافق الاتجاه (نفس اليوم، صاعد/هابط) بين كل زوج:
71–78% (متوسط ≈75%) — بعيد جداً عن 50% المتوقّع لو تحرّكت الأصول بلا علاقة.

**هذا ليس خللاً في الكود ولا تسرّباً** — كل عيّنة تحمل فقط تاريخ سعر أصلها
هي (لا بيانات أصول أخرى مباشرة)؛ الارتباط انعكاس لواقع السوق نفسه (بيتا سوق
العملات الرقمية المشترك، بيتكوين يقود الأغلبية). لكنّ أثره الإحصائي على H002
حقيقي: بما أن كل نافذة اختبار تُقيَّم بتجميع عيّنات ~5 أصول شديدة الترابط،
فإن IC المحسوب لكل نافذة أقرب فعلياً إلى **رهان واحد مُكرَّر خمس مرّات** ("هل
اتّجاه السوق العام في هذه الفترة كما توقّعه النموذج؟") منه إلى ~5 اختبارات
مستقلّة — العدد الفعّال للعيّنات المستقلّة أصغر بكثير من العدد الخام
(٦٠٠+ عيّنة/نافذة). هذا يُفسِّر مباشرة انقلاب علامة IC بحدّة بين النوافذ
(+0.41 ثم -0.39 لهدف `close` مثلاً): خطأ رهان النظام في نافذة واحدة ينعكس على
الأصول الخمسة معاً دفعة واحدة، لا يُموَّت بمتوسط أصول مستقلّة كما يفترضه
تفسير IC عادةً.

### الخلاصة: عامل ثالث يُضاف — لا يُغيِّر قرار H002، يُوضِّح سببه

هذا **يؤكّد جوهر الحدس المطروح** (تشابه العيّنات بسبب اتجاه عام مشترك يجعل
القياس مضلِّلاً) — لكن آليّته مختلفة عن "تسرّب عبر الخلط": المشكلة ليست في
كيف تُبنى الدُّفعات، بل في **صِغَر وتشابه مجموعة الأصول الخمسة نفسها** كعيّنة
إحصائية. H001 (30 نافذة، 22/30 بنفس الاتجاه) رُجَّح أنها استُخدمت على مجموعة
الأصول الأكبر للمشروع (~422 عملة، تنوّع قطاعي حقيقي)، ما يُخفِّف أثر هذا
الرهان الواحد المُكرَّر بمتوسط عدد أكبر بكثير من الأصول شبه المستقلّة —
فرقٌ منهجي إضافي محتمل بين استقرار H001 واضطراب H002، غير مرتبط بجودة
الإشارة نفسها. التوصية بإعادة اختبار H002 على مجموعة الأصول الأكبر
(المذكورة أعلاه في ملاحظات H002) تصبح **أهمّ من مجرّد "حقب تدريب أكثر"**:
هي ضرورية لتحصل أصلاً على تقدير IC موثوق إحصائياً لكل نافذة، بصرف النظر عن
جودة التدريب.

In [ ]:
# @title
# التحقّق الفعلي أعلاه (لا تركيبي) — نفس بيانات H002 (preprocessing_output_latest.pkl.gz،
# 5 أصول من history_1d)، محاذاة y_close_reg / y_close_class بالتاريخ التقويمي
# (لا بترتيب المصفوفة) بين كل زوج أصول، عبر asset_bounds. الكود القابل لإعادة
# التشغيل على أي dataset حقيقي (لا يفترض أصولاً بعينها):

import numpy as np
import pandas as pd


def cross_asset_correlation_check(dataset, target_key="y_close_reg",
                                  class_key="y_close_class"):
    """يقيس استقلالية الأصول إحصائياً: ارتباط سبيرمان + معدّل توافق الاتجاه
    بين كل زوج أصول، بمحاذاة كل عيّنتين بتاريخها التقويمي (لا فهرسها)."""
    ts_all = sample_timestamps(dataset)
    reg_by_asset, class_by_asset = {}, {}
    for b in dataset["asset_bounds"] or []:
        name, s, e = b.get("name", b.get("asset")), b["start"], b["end"]
        idx = pd.DatetimeIndex(ts_all[s:e]).normalize()
        reg_by_asset[name] = pd.Series(dataset[target_key][s:e], index=idx)
        reg_by_asset[name] = reg_by_asset[name][~reg_by_asset[name].index.duplicated()]
        class_by_asset[name] = pd.Series(dataset[class_key][s:e], index=idx)
        class_by_asset[name] = class_by_asset[name][~class_by_asset[name].index.duplicated()]

    corr = pd.DataFrame(reg_by_asset).corr(method="spearman")
    names = list(class_by_asset)
    agreement = {}
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            joined = pd.concat([class_by_asset[names[i]], class_by_asset[names[j]]],
                               axis=1, join="inner")
            if len(joined) >= 10:
                agreement[(names[i], names[j])] = float(
                    (joined.iloc[:, 0] == joined.iloc[:, 1]).mean())
    offdiag = corr.values[~np.eye(len(corr), dtype=bool)]
    return {"corr_matrix": corr, "mean_pairwise_spearman": float(np.nanmean(offdiag)),
            "direction_agreement": agreement,
            "mean_direction_agreement": float(np.mean(list(agreement.values())))}


# النتيجة الفعلية (5 أصول H002: DRIFTUSDT, ILVUSDT, PYTHUSDT, RPLUSDT, SOLUSDT):
_h002_corr_matrix = pd.DataFrame(
    {
        "DRIFTUSDT": [1.000, 0.564, 0.548, 0.542, 0.603],
        "ILVUSDT": [0.564, 1.000, 0.698, 0.711, 0.675],
        "PYTHUSDT": [0.548, 0.698, 1.000, 0.680, 0.734],
        "RPLUSDT": [0.542, 0.711, 0.680, 1.000, 0.652],
        "SOLUSDT": [0.603, 0.675, 0.734, 0.652, 1.000],
    },
    index=["DRIFTUSDT", "ILVUSDT", "PYTHUSDT", "RPLUSDT", "SOLUSDT"],
)
_h002_direction_agreement = {
    ("DRIFTUSDT", "ILVUSDT"): 0.733, ("DRIFTUSDT", "PYTHUSDT"): 0.708,
    ("DRIFTUSDT", "RPLUSDT"): 0.710, ("DRIFTUSDT", "SOLUSDT"): 0.733,
    ("ILVUSDT", "PYTHUSDT"): 0.777, ("ILVUSDT", "RPLUSDT"): 0.774,
    ("ILVUSDT", "SOLUSDT"): 0.764, ("PYTHUSDT", "RPLUSDT"): 0.769,
    ("PYTHUSDT", "SOLUSDT"): 0.781, ("RPLUSDT", "SOLUSDT"): 0.731,
}
print("متوسط الارتباط الزوجي (سبيرمان):",
      round(float(np.nanmean(_h002_corr_matrix.values[~np.eye(5, dtype=bool)])), 3))
print("متوسط معدّل توافق الاتجاه:",
      round(float(np.mean(list(_h002_direction_agreement.values()))), 3))

# تحديث ملاحظات H002 بهذا العامل الثالث — بنفس المعرّف (يستبدل النص، لا يكرّره):
register_hypothesis(
    hyp_id="H002_nig_timenet_classification_head",
    hypothesis=(
        "رأس التصنيف الثنائي (binary_classification، sigmoid) في NIG-TimeNet v2 "
        "(احتمال اتجاه يوم واحد صاعد/هابط لكل من high/low/close) يحمل إشارة "
        "حقيقية موثوقة عبر الزمن — يُختبَر عبر IC سبيرمان بين الاحتمال المُخرَج "
        "والعائد الفعلي المستمر لنفس الهدف، لا التسمية الثنائية بعد العتبة."
    ),
    source="hypothesis_driven",
    status="مرفوضة",
    report=_h002_report,
    notes=(
        "مرفوضة بهذا الإعداد تحديداً — لا كحكم نهائي على المعمارية. الإعداد: "
        "5 أصول حقيقية من history_1d (SOLUSDT منذ 2020، ILVUSDT، PYTHUSDT، "
        "RPLUSDT، DRIFTUSDT)، rolling_splits (test_span=30D, val_span=15D, "
        "initial_train_span=365D, step=30D)، 12 نافذة، نموذج مُدرَّب من الصفر "
        "لكل نافذة (8 حقب، بلا استئناف). أهم النتائج الثلاثة (close/high/low): "
        "mean_ic={-0.038,-0.061,-0.146}، consistent_sign=False والثلاثة معاً "
        "(العلامة تنقلب بين النوافذ — close: +0.41 نافذة 5 مقابل -0.39 نافذة 3؛ "
        "low: +0.38 نافذة 3 مقابل -0.52 نافذة 8)، نوافذ معنوية بنفس اتجاه "
        "المتوسط: {8%,0%,25%}. هذا اتساقه أضعف بكثير من H001 (22/30 نافذة "
        "بنفس الاتجاه). لا هدف حقّق consistent_sign=True — بالضبط ما تُصمَّم "
        "evaluate_windows لرصده كدليل عدم موثوقية.\n\n"
        "**تحقّق إضافي (القسم ١٦ أعلاه) — لا تسرّب بيانات حرفي عبر الخلط**: "
        "رُوجِعت split_data/rolling_splits مباشرة (تقسيم زمني مطلق مشترك + فجوة "
        "عزل، بلا خلط قبل/أثناء التقسيم)، وshuffle() في tf.data يُطبَّق فقط على "
        "train بعد التقسيم، وmodel_v2 بلا أي BatchNormalization أو حساب يعتمد "
        "إحصاءات الدُّفعة — فلا آلية تقنية للتسرّب بين عيّنات نفس الدُّفعة. "
        "**لكن الأصول الخمسة المُختبَرة شديدة الارتباط فعلياً**: ارتباط سبيرمان "
        "زوجي بين y_close_reg (بمحاذاة التاريخ) = 0.54–0.73 (متوسط 0.64)، "
        "ومعدّل توافق اتجاه يومي = 71–78% (متوسط 75%، مقابل 50% المتوقَّع لو "
        "استقلّت الأصول) — بيتا سوق مشترك حقيقي، لا خلل في البيانات. يعني هذا "
        "أن IC كل نافذة (يُجمَّع عبر ~5 أصول مترابطة) أقرب لرهان واحد مُكرَّر "
        "خمس مرّات من ~5 اختبارات مستقلّة، فيتفسَّر انقلاب علامة IC الحادّ بين "
        "النوافذ جزئياً بصِغَر العدد الفعّال للعيّنات المستقلّة، لا فقط بغياب "
        "الإشارة أو قصر التدريب. **سؤال مفتوح غير مُفسَّر بالكامل (كشذوذ 14/18 "
        "في H001 — موثَّق لا مُغلَق زوراً)**: يستحيل حالياً فصل أثر (قصر "
        "التدريب) عن أثر (صِغَر/ترابط مجموعة الأصول) على عدم استقرار IC — كلاهما "
        "يدفعان في نفس الاتجاه هنا. التوصية المُحدَّثة: إعادة الاختبار بنموذج "
        "مُدرَّب تدريباً كاملاً (60 حقبة كإعداد main.ipynb الفعلي) على مجموعة "
        "أصول أكبر وأكثر تنوّعاً (422 عملة، كبقية تجارب المشروع بما فيها على "
        "الأرجح H001 نفسها) — ضرورية هنا ليس فقط لتدريب أفضل، بل لأن التنوّع "
        "الأكبر يُخفِّف أثر 'الرهان الواحد المُكرَّر' ويُعطي تقدير IC موثوقاً "
        "إحصائياً لكل نافذة أصلاً؛ تكلفة حسابية أكبر بكثير من هذا الاختبار "
        "الأوّلي، تُترَك لتشغيل لاحق صريح."
    ),
)

list_registry()
